In [1]:
import pandas as pd
import numpy as np

f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

training = pd.concat([kazuki_data, yokoi_data.loc[0,:].iloc[:2000], yokoi_data.loc[1,:].iloc[:200]])
validation = pd.concat([watara_data, yokoi_data.loc[0,:].iloc[2000:], yokoi_data.loc[1,:].iloc[200:]])

t_healthy = training.loc[0]
t_cancer = training.loc[1]
v_healthy = validation.loc[0]
v_cancer = validation.loc[1]

m_t_healthy = t_healthy.shape[0]
m_t_cancer = t_cancer.shape[0]
m_v_healthy = v_healthy.shape[0]
m_v_cancer = v_cancer.shape[0]

n_mirnas = t_healthy.shape[1] #gets number of columns
miRNA_names_list = list(t_healthy.columns)

print("healthy training data")
print(t_healthy.shape)
print(t_healthy.head)
print("cancer training data")
print(t_cancer.shape)
print(t_cancer.head)
print("healthy validation data")
print(v_healthy.shape)
print(v_healthy.head)
print("cancer validation data")
print(v_cancer.shape)
print(v_cancer.head)

healthy training data
(6965, 2565)
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  MIMAT0000064  MIMAT0000065  MIMAT0000066  \
0          3.700         3.669        -2.681         3.499        -2.681   
0         -4.018        -4.018        -4.018        -4.018        -4.018   
0          1.514         0.869         0.464        -0.987        -4.325   
0         -4.417        -4.417        -4.417        -1.036        -1.897   
0         -5.541         0.598        -5.281        -3.185         0.986   
..           ...           ...           ...           ...           ...   
0         -3.515         1.626         6.450        -3.515         1.136   
0         -3.160        -3.160        -3.160        -3.160        -3.160   
0         -3.717         0.509        -3.717        -3.717         1.166   
0         -2.374        -2.374         1.426         2.128         3.553   
0         -5.741         1.007        -5.741        -5.741        -5.741   

    MIMAT0000067  MIMA

In [2]:
Y = t_healthy.values
healthy_comp = np.zeros((n_mirnas, n_mirnas))
for a in range(n_mirnas):
    for b in range(n_mirnas):
        healthy_comp[a,b]=np.sum(Y[:,a] > Y[:,b])
healthy_comp = healthy_comp / m_t_healthy
print(healthy_comp)

[[0.         0.22368988 0.33725772 ... 0.13639627 0.41005025 0.42857143]
 [0.54515434 0.         0.57501795 ... 0.18880115 0.63460158 0.66245513]
 [0.29605169 0.20904523 0.         ... 0.11500359 0.4011486  0.42038765]
 ...
 [0.82311558 0.77774587 0.83833453 ... 0.         0.89375449 0.90193826]
 [0.19109835 0.14630294 0.20215363 ... 0.04005743 0.         0.25886576]
 [0.10150754 0.06661881 0.10911701 ... 0.03603733 0.15046662 0.        ]]


In [3]:
CONST_k = 0.95
stable_pairs = np.where((healthy_comp >= CONST_k) | (healthy_comp <= 1-CONST_k), 
                        1, 0)

In [4]:
print(stable_pairs.nonzero()[0].shape)

(2563428,)


In [5]:
Z = t_cancer.values
cancer_comp = np.zeros((n_mirnas, n_mirnas))
for a in range(n_mirnas):
    for b in range(n_mirnas):
        cancer_comp[a,b]=np.sum(Z[:,a] > Z[:,b])
cancer_comp = cancer_comp / m_t_cancer
print(cancer_comp)

[[0.    0.45  0.525 ... 0.505 0.67  0.635]
 [0.4   0.    0.505 ... 0.475 0.645 0.665]
 [0.29  0.31  0.    ... 0.38  0.52  0.49 ]
 ...
 [0.4   0.43  0.455 ... 0.    0.59  0.56 ]
 [0.15  0.13  0.18  ... 0.145 0.    0.205]
 [0.14  0.125 0.215 ... 0.23  0.385 0.   ]]


In [6]:
CONST_l = 0.75
stable_cancer_pairs = np.where((cancer_comp >= CONST_l) | (cancer_comp <= 1-CONST_l), 
                        1, 0)

In [7]:
identity_matrix = np.identity(n_mirnas,dtype=bool)
candidate_diagnostic_pairs = (((stable_pairs) & (stable_cancer_pairs)) & 
                            ((np.absolute(healthy_comp - cancer_comp)) > 0.5) &
                            (~identity_matrix))



In [8]:
'''
you might consider getting rid of redundnacy here.
check for correlation of clusters
'''
selected = candidate_diagnostic_pairs.nonzero()
# double check ts actually passes 
# for i in range(2):
#     for j in range(len(selected[i])):
#         print(miRNA_names_list[selected[i][j]] in og_set1 or miRNA_names_list[selected[i][j]] in og_set2)
# for i in range(len(selected[0])):
#     x = (t_healthy.iloc[:, selected[0][i]] > t_healthy.iloc[:, selected[1][i]]).sum()
#     print(f'mirna {selected[0][i]} and {selected[1][i]}')
print(selected[0].shape)
counts = np.bincount(selected[0])
for i, count in enumerate(counts):
    if count > 0:
        print(f"{i} occurs {count} times")


(235,)
81 occurs 4 times
135 occurs 10 times
148 occurs 3 times
167 occurs 3 times
334 occurs 1 times
345 occurs 3 times
430 occurs 1 times
489 occurs 1 times
535 occurs 1 times
542 occurs 2 times
573 occurs 1 times
589 occurs 1 times
614 occurs 1 times
668 occurs 1 times
673 occurs 4 times
688 occurs 3 times
697 occurs 1 times
713 occurs 12 times
714 occurs 4 times
730 occurs 1 times
731 occurs 1 times
732 occurs 1 times
745 occurs 6 times
765 occurs 1 times
791 occurs 1 times
836 occurs 1 times
873 occurs 1 times
931 occurs 1 times
957 occurs 2 times
965 occurs 2 times
966 occurs 2 times
970 occurs 1 times
979 occurs 1 times
980 occurs 1 times
1028 occurs 5 times
1050 occurs 1 times
1100 occurs 1 times
1123 occurs 1 times
1202 occurs 1 times
1206 occurs 1 times
1246 occurs 1 times
1298 occurs 1 times
1303 occurs 1 times
1319 occurs 1 times
1326 occurs 1 times
1430 occurs 1 times
1463 occurs 1 times
1477 occurs 1 times
1496 occurs 3 times
1499 occurs 3 times
1508 occurs 1 times
1571 o

In [9]:
'''generate input'''
pair_dict_training = {}
for i in range(len(selected[0])):
    _key = f"{selected[0][i]},{selected[1][i]}"
    _healthy_sect = Y[:,selected[0][i]]> Y[:,selected[1][i]]
    _cancer_sect = Z[:,selected[0][i]]> Z[:,selected[1][i]]
    pair_dict_training[_key] = np.concat((_healthy_sect, _cancer_sect))
_index = np.concat((np.zeros(m_t_healthy),np.ones(m_t_cancer)))
X_unshuffled = pd.DataFrame(pair_dict_training, index = _index)
X = X_unshuffled.sample(frac=1, random_state=42)
y = np.array(list(X.index))
X = X.reset_index().drop('index', axis=1)

In [10]:
X.shape

(7165, 235)

In [11]:
np.sum(y)

np.float64(200.0)

In [27]:
#logistic regression with lasso
import matplotlib.pyplot as plt
import numpy as np

from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats
# not importing scaling bc all of it is binary

In [33]:
#increase tolerance for short training time ? idk what ts means
svc = SVC(C=1.0, class_weight=None, kernel='linear')
'''sklearn.svm.SVC(kernel = 'poly', degree=3), #default
sklearn.svm.SVC(kernel = 'rbf'),
sklearn.svm.SVC(kernel = 'sigmoid')'''
param_grid = {'C': scipy.stats.expon(scale=100), 
               'class_weight':['balanced',None]}
RandomSearchCV = RandomizedSearchCV(estimator=svc, param_distributions=param_grid, 
                                    n_iter=50, random_state=42)
RandomSearchCV.fit(X,y)
best_params = RandomSearchCV.get_params()
print(best_params) #i just changed the og svc to the params given

{'cv': None, 'error_score': nan, 'estimator__C': 1.0, 'estimator__break_ties': False, 'estimator__cache_size': 200, 'estimator__class_weight': None, 'estimator__coef0': 0.0, 'estimator__decision_function_shape': 'ovr', 'estimator__degree': 3, 'estimator__gamma': 'scale', 'estimator__kernel': 'linear', 'estimator__max_iter': -1, 'estimator__probability': False, 'estimator__random_state': None, 'estimator__shrinking': True, 'estimator__tol': 0.001, 'estimator__verbose': False, 'estimator': SVC(kernel='linear'), 'n_iter': 50, 'n_jobs': None, 'param_distributions': {'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x1409a6b70>, 'class_weight': ['balanced', None]}, 'pre_dispatch': '2*n_jobs', 'random_state': 42, 'refit': True, 'return_train_score': False, 'scoring': None, 'verbose': 0}


In [ ]:
model = RFE(estimator=svc, n_features_to_select=6, step=1)
model.fit(X,y)

y_pred = model.decision_function(X)
y_prediction_values = np.array(y_pred)
cm = confusion_matrix(y, y_prediction_values>0)
sensitivity = cm[1,1]/(cm[1,1]+cm[0,1])
specificity = (cm[0,0]/(cm[0,0]+cm[1,0]))
print("Confusion Matrix: \n", cm)
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(classification_report(y,y_prediction_values>0))

Confusion Matrix: 
 [[6949   16]
 [   8  192]]
Sensitivity (Recall): 0.9231
Specificity: 0.9989
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      6965
         1.0       0.92      0.96      0.94       200

    accuracy                           1.00      7165
   macro avg       0.96      0.98      0.97      7165
weighted avg       1.00      1.00      1.00      7165



In [ ]:
#get validation data
'''Y, Z are healthy/ovarian training data'''
'''im going to make W,X healthy/ovarian validation'''
